# Событийное исследование: цена

Оценка рыночной модели, накопленная аномальная доходность и проверка
результата плацебо-тестом. Запускается после `make analysis`.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from src import moex, pipeline
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
P = moex.DATA_PROCESSED

## Модель рынка

Альфа и бета оцениваются на окне [−250, −40]. Правая граница не −1:
о ребалансировке объявляют примерно за две недели, и реакция на объявление
не должна попасть в оценку беты.

In [2]:
data = pipeline.load()
parameters = data.parameters.dropna(subset=['beta'])
parameters[['beta', 'alpha', 'r_squared', 'n_estimation']].describe().round(3)

,beta,alpha,r_squared,n_estimation
count,37.000,37.000,37.000,37.000
mean,0.944,-0.001,0.254,201.216
std,0.323,0.002,0.105,26.142
min,0.325,-0.005,0.065,85.000
25%,0.726,-0.002,0.179,210.000
50%,0.934,-0.000,0.242,211.000
75%,1.207,0.001,0.342,211.000
max,1.782,0.004,0.438,211.000


## Накопленная аномальная доходность по окнам

In [3]:
results = pd.read_parquet(P / 'car_results.parquet')
results[['event_type', 'window', 'n_events', 'mean_car', 'median_car',
         'share_positive', 'p_naive', 'p_clustered', 'p_holm', 'n_clusters']].round(4)

,event_type,window,n_events,mean_car,median_car,share_positive,p_naive,p_clustered,p_holm,n_clusters
0,включение,"[-30,-1] до события",14,0.0229,0.0242,0.6429,0.3362,0.8489,1.0000,10
1,включение,"[-1,+1] ребалансировка",14,0.0054,0.0100,0.7857,0.6931,0.8128,1.0000,10
2,включение,"[+1,+20] ближний откат",14,-0.0876,-0.0775,0.0714,0.0177,0.0719,NaN,10
3,включение,"[+1,+60] дальний откат",14,-0.1830,-0.2421,0.1429,0.0004,0.0028,0.0169,10
4,включение,"[-30,+60] всё окно",14,-0.1626,-0.2008,0.1429,0.0028,0.0073,NaN,10
5,исключение,"[-30,-1] до события",12,0.0135,0.0082,0.5833,0.7195,0.8520,1.0000,6
6,исключение,"[-1,+1] ребалансировка",12,-0.0007,-0.0144,0.2500,0.9632,0.4430,1.0000,6
7,исключение,"[+1,+20] ближний откат",12,0.0340,0.0374,0.5833,0.2775,0.1178,NaN,6
8,исключение,"[+1,+60] дальний откат",12,0.0387,-0.0053,0.5000,0.2867,0.2383,1.0000,6
9,исключение,"[-30,+60] всё окно",12,0.0584,0.0372,0.5833,0.2209,0.1372,NaN,6


Обратите внимание на расхождение `p_naive` и `p_clustered`: наивный тест
считает события независимыми, хотя 42 события приходятся на 21 дату.

## Вклад отдельных бумаг

In [4]:
from src import abnormal_returns as arm
abnormal = pd.read_parquet(P / 'abnormal_returns.parquet')

for event_type in ['включение', 'исключение']:
    car = arm.car_by_event(abnormal[abnormal.event_type == event_type], (1, 60))
    car = car.sort_values('car')
    print(f'--- {event_type}: среднее {car.car.mean():+.1%}, медиана {car.car.median():+.1%} ---')
    print('  ' + '  '.join(f'{r.ticker}:{r.car:+.0%}' for r in car.itertuples()))
    print(f'  без двух крайних значений: {car.car.iloc[1:-1].mean():+.1%}')
    print()

--- включение: среднее -18.3%, медиана -24.2% ---
  SELG:-40%  FLOT:-32%  UGLD:-30%  RAGR:-30%  LENT:-27%  POSI:-27%  ASTR:-27%  SMLT:-22%  RENI:-13%  MDMG:-9%  CNRU:-8%  X5:-3%  SVCB:+4%  SGZH:+7%
  без двух крайних значений: -18.6%

--- исключение: среднее +3.9%, медиана -0.5% ---
  ASTR:-19%  LEAS:-5%  PIKK:-3%  MTLRP:-3%  MGNT:-2%  HYDR:-1%  UPRO:+0%  MTLR:+11%  SELG:+11%  FEES:+16%  SMLT:+20%  SGZH:+21%
  без двух крайних значений: +4.4%



Эффект у включений не создаётся выбросами: удаление двух крайних значений
почти не меняет среднее.

## Плацебо-тест

Основной критерий вывода. Кластерная асимптотика на 6–10 кластерах не
работает, а рандомизационный вывод не требует ни нормальности, ни
асимптотики.

In [5]:
placebo_results = pd.read_parquet(P / 'placebo_results.parquet')
placebo_results[['event_type', 'window', 'observed', 'percentile',
                 'p_value', 'placebo_mean', 'placebo_std']].round(4)

,event_type,window,observed,percentile,p_value,placebo_mean,placebo_std
0,включение,"[-30,-1] до события",0.0229,72.7,0.6324,-0.0081,0.0521
1,включение,"[-1,+1] ребалансировка",0.0054,69.3,0.6923,-0.0010,0.0172
2,включение,"[+1,+60] дальний откат",-0.1830,3.8,0.0390,-0.0313,0.0792
3,исключение,"[-30,-1] до события",0.0135,71.4,0.8042,-0.0154,0.0503
4,исключение,"[-1,+1] ребалансировка",-0.0007,53.5,0.9500,-0.0020,0.0147
5,исключение,"[+1,+60] дальний откат",0.0387,87.4,0.6154,-0.0392,0.0689


`placebo_mean` для окна [+1, +60] заметно отрицателен. Альфа, оценённая на
прошлом окне, систематически не выполняется вперёд, и процедура сама по себе
выдаёт небольшую отрицательную доходность. Наивный тест этого не видит.

In [6]:
window = (1, 60)
for event_type in ['включение', 'исключение']:
    path = P / f'placebo_{event_type}_{window[0]}_{window[1]}.parquet'
    distribution = pd.read_parquet(path)['placebo_car'].dropna()
    row = placebo_results[(placebo_results.event_type == event_type) &
                          (placebo_results.window == '[+1,+60] дальний откат')].iloc[0]
    print(f'{event_type}: факт {row.observed:+.2%}, '
          f'плацебо {distribution.mean():+.2%} ± {distribution.std():.2%}, '
          f'перцентиль {row.percentile:.1f}, p = {row.p_value:.3f}')

включение: факт -18.30%, плацебо -3.13% ± 7.92%, перцентиль 3.8, p = 0.039
исключение: факт +3.87%, плацебо -3.92% ± 6.89%, перцентиль 87.4, p = 0.615
